# Extract DEX prices from Dune

Pull swap-level mid prices for one token pair on one blockchain, from each supported DEX, over a **collection window**, then save one CSV per DEX under `<chain>/`.

All reusable logic lives in the `arblib` package; this notebook only sets parameters and wires the steps together.

In [1]:
# !pip install -r requirements.txt

In [2]:
from arblib.config import SWAP_QUERY_IDS, LIQUIDITY_QUERY_IDS, TOKENS, build_collection_params
from arblib.dune_api import make_headers, run_dune_saved_query
from arblib.data_io import save_dataframes
import os
from pathlib import Path


## Parameters

- **CHAIN / tokens** — which market to pull.
- **Collection window** (`START_TS` / `END_TS`, UTC) — note `START_TS` is deliberately *earlier* than the study start used in `arbitrage.ipynb`, so every pool already has a known price to forward-fill from once the study window begins.

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  
DUNE_API_KEY = os.environ["DUNE_API_KEY"]

current = Path.cwd()
while current.name != 'defi_arbitrage' and current != current.parent:
    current = current.parent
BASE_DIR = current if current.name == 'defi_arbitrage' else Path.cwd()

# --- What to collect ------------------------------------------------
CHAIN  = "ethereum"                 # blockchain name
TOKEN0 = TOKENS[CHAIN]["WETH"]  # base token
TOKEN1 = TOKENS[CHAIN]["USDC"]  # quote token

# --- Collection window (UTC) ---------------------------------------
START_TS = "2025-12-31 15:00:00"
END_TS   = "2025-12-31 16:00:00"

# END_TS   = "2025-01-01 12:00:00"


params  = build_collection_params(CHAIN, TOKEN0, TOKEN1, START_TS, END_TS)
headers = make_headers(DUNE_API_KEY)
params

{'start_ts': '2025-12-31 15:00:00',
 'end_ts': '2025-12-31 16:00:00',
 'token0': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
 'token1': '0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48',
 'chain': 'ethereum'}

## Run the swap queries

Swap-level mid prices, one query per DEX. A DEX not available on `CHAIN` returns an empty DataFrame and is skipped on save.

In [4]:
df_uniswap_swap   = run_dune_saved_query(SWAP_QUERY_IDS["uniswap"],   params, headers, "Uniswap")
df_pancake_swap   = run_dune_saved_query(SWAP_QUERY_IDS["pancake"],   params, headers, "Pancake")


[Uniswap] EXECUTE RESPONSE: {'execution_id': '01KV5Y4RBFKD05M1ZRZRPZPC45', 'state': 'QUERY_STATE_PENDING'}
[Uniswap] STATUS: QUERY_STATE_PENDING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_EXECUTING
[Uniswap] STATUS: QUERY_STATE_COMPLETED
[Pancake] EXECUTE RESPONSE: {'execution_id': '01KV5Y5D4N2D80HENE9N3NYF72', 'state': 'QUERY_STATE_PENDING'}
[Pancake] STATUS: QUERY_STATE_PENDING
[Pancake] STATUS: QUERY_STATE_EXECUTING
[Pancake] STATUS: QUERY_STATE_COMPLETED


## Run the liquidity queries

Mint / burn events per pool, one query per DEX, over the same `params` window. Used downstream to reconstruct the liquidity state at any block.

In [5]:
df_uniswap_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["uniswap"], params, headers, "Uniswap liquidity")
df_pancake_liq = run_dune_saved_query(LIQUIDITY_QUERY_IDS["pancake"], params, headers, "Pancake liquidity")


[Uniswap liquidity] EXECUTE RESPONSE: {'execution_id': '01KV5Y5SEG3QG19AYH6DNBY28P', 'state': 'QUERY_STATE_PENDING'}
[Uniswap liquidity] STATUS: QUERY_STATE_PENDING
[Uniswap liquidity] STATUS: QUERY_STATE_EXECUTING
[Uniswap liquidity] STATUS: QUERY_STATE_COMPLETED
[Pancake liquidity] EXECUTE RESPONSE: {'execution_id': '01KV5Y66EEHVZPVG0D9KR8V7SS', 'state': 'QUERY_STATE_PENDING'}
[Pancake liquidity] STATUS: QUERY_STATE_PENDING
[Pancake liquidity] STATUS: QUERY_STATE_EXECUTING
[Pancake liquidity] STATUS: QUERY_STATE_COMPLETED
[INFO] Pancake liquidity: no data returned


## Save the swaps under `<chain>/swaps/`

In [6]:
swap_dir = os.path.join(BASE_DIR, CHAIN, "swaps")

save_dataframes(
    {
        "df_uniswap_swap.csv": df_uniswap_swap,
        "df_pancake_swap.csv": df_pancake_swap,
    },
    swap_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_uniswap_swap.csv
Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/swaps/df_pancake_swap.csv
Done.


## Save the liquidity under `<chain>/liquidity/`

In [7]:
liquidity_dir = os.path.join(BASE_DIR, CHAIN, "liquidity")

save_dataframes(
    {
        "df_uniswap_liq.csv": df_uniswap_liq,
        "df_pancake_liq.csv": df_pancake_liq,
    },
    liquidity_dir,
)

Saved: /Users/matthieu/Downloads/DeFi_limits-to-arbitrage/ethereum/liquidity/df_uniswap_liq.csv
Skipped empty or missing dataframe: df_pancake_liq.csv
Done.
